# spaDIVA tutorial: unique factor ratio on simulated data

This tutorial gives a complete, self-contained example of feature-level unique factor ratio analysis using the simulated data included with spaDIVA.

## 1. Import packages

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch
from sklearn.decomposition import PCA

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
from spaDIVA import (
    cal_spatial,
    clr_normalize_each_cell,
    cluster,
    collect_spadiva_outputs,
    correlate_metrics,
    evaluate_unique_factor_ratio,
    featurewise_pearson,
    fuse_shared_latent,
    infer_latents,
    ranked_rolling_mean,
    ridge_predict,
    select_shared_latent,
    summarize_extreme_groups,
    train_spadiva,
)

sc.set_figure_params(figsize=(3, 3))
plt.rcParams["figure.dpi"] = 120

## 2. Set run options

In [ ]:
RANDOM_SEED = 42
POE_SAMPLE_SEED = RANDOM_SEED
USE_CUDA = False
MAX_EPOCHS = 200

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

## 3. Load simulated paired modalities

Omics 1 is the ADT-like modality and omics 2 is the RNA-like modality.

In [ ]:
DATA_DIR = PROJECT_ROOT / "examples" / "simulation_data"
adata_omics1 = sc.read(DATA_DIR / "ADT_100.h5ad")
adata_omics2 = sc.read(DATA_DIR / "RNA_ZINB.h5ad")

adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()
adata_omics1.X = adata_omics1.layers["counts"].copy()
adata_omics2.X = adata_omics2.layers["counts"].copy()
adata_omics1.X = adata_omics1.X.astype("float32")
adata_omics2.X = adata_omics2.X.astype("float32")

adata_omics1, adata_omics2

## 4. Preprocess modalities

The ADT-like modality is CLR-normalized and the RNA-like modality is log-normalized. Both modalities are reduced to 64 principal components for spaDIVA training. The normalized ADT-like matrix is kept as the target feature matrix for the unique factor ratio analysis.

In [ ]:
adata_omics1 = clr_normalize_each_cell(adata_omics1)
sc.pp.scale(adata_omics1)
Y_target = np.asarray(adata_omics1.X, dtype=np.float32)
feature_names = np.asarray(adata_omics1.var_names).astype(str)

sc.pp.normalize_total(adata_omics2, target_sum=1e4)
sc.pp.log1p(adata_omics2)
sc.pp.scale(adata_omics2)

pca1 = PCA(n_components=64, random_state=RANDOM_SEED)
adata_omics1.obsm["pca"] = pca1.fit_transform(adata_omics1.to_df())

pca2 = PCA(n_components=64, random_state=RANDOM_SEED)
adata_omics2.obsm["pca"] = pca2.fit_transform(adata_omics2.to_df())

## 5. Build the spatial graph and train spaDIVA

In [ ]:
spatial = adata_omics1.obsm["spatial"]
edge_index = cal_spatial(spatial, k=4)

X1_input = adata_omics1.obsm["pca"]
X2_input = adata_omics2.obsm["pca"]

model, train_loss = train_spadiva(
    X1_input,
    X2_input,
    X1_input,
    X2_input,
    edge_index=edge_index,
    learning_rate=1e-3,
    weight=1.0,
    max_epochs=MAX_EPOCHS,
    use_cuda=USE_CUDA,
)

## 6. Infer spaDIVA representations

In [ ]:
Z_poe, Z1_loc, Z2_loc, W1_loc, W2_loc, X1_hat, X2_hat = infer_latents(
    model,
    X1_input,
    X2_input,
    edge_index=edge_index,
    use_cuda=USE_CUDA,
    sample_seed=POE_SAMPLE_SEED,
)

Z_loc = fuse_shared_latent(Z1_loc, Z2_loc, k=20)
z_adata = collect_spadiva_outputs(
    Z=Z_loc,
    W1=W1_loc,
    W2=W2_loc,
    spatial=spatial,
    obs_names=adata_omics1.obs_names,
    modality_names=("omics1", "omics2"),
    Z_poe=Z_poe,
    Z1=Z1_loc,
    Z2=Z2_loc,
    X1_hat=X1_hat,
    X2_hat=X2_hat,
    model_seed=RANDOM_SEED,
    poe_sample_seed=POE_SAMPLE_SEED,
)

z_adata

## 7. Create a lightweight external prediction

The unique factor ratio API can optionally be compared with an external cross-modality prediction. Here, a simple ridge model predicts omics 1 features from omics 2 PCA features. This tutorial uses ridge regression to demonstrate the external-prediction interface; the manuscript simulation analysis used saved totalVI predictions.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
all_idx = np.arange(adata_omics1.n_obs)
rng.shuffle(all_idx)
n_train = int(0.7 * len(all_idx))
train_idx = np.sort(all_idx[:n_train])
test_idx = np.sort(all_idx[n_train:])

external_pred = ridge_predict(
    X2_input[train_idx],
    Y_target[train_idx],
    X2_input[test_idx],
    alpha=10.0,
)

external_metric = featurewise_pearson(Y_target[test_idx], external_pred)
pd.Series(external_metric, name="ridge_RNA_to_ADT_pearson").describe()

## 8. Compute feature-level unique factor ratio

For each omics 1 feature, spaDIVA compares how well the selected shared representation predicts the feature with how well its combination with `W_omics1` predicts it. The default selector uses `Z_poe`; if `Z_poe` is unavailable it falls back to `Z`, while `use="Z"` explicitly selects the WNN representation. The unique factor ratio summarizes the additional contribution of the modality-specific representation.

In [ ]:
Z_shared, z_source = select_shared_latent(z_adata)

unique_factor_df = evaluate_unique_factor_ratio(
    Z_train=Z_shared[train_idx],
    W_train=z_adata.obsm["W_omics1"][train_idx],
    Y_train=Y_target[train_idx],
    Z_test=Z_shared[test_idx],
    W_test=z_adata.obsm["W_omics1"][test_idx],
    Y_test_true=Y_target[test_idx],
    external_pred=external_pred,
    feature_names=feature_names,
    external_metric_name="ridge_RNA_to_ADT_pearson",
    ridge_alpha=1.0,
    r2_threshold=0.001,
    w_key="W_omics1",
    z_source=z_source,
    model_seed=z_adata.uns["model_seed"],
    poe_sample_seed=(
        z_adata.uns.get("poe_sample_seed") if z_source in ("Z_poe", "Z_PoE") else None
    ),
)

corr_summary = correlate_metrics(
    unique_factor_df["unique_factor_ratio"],
    unique_factor_df["ridge_RNA_to_ADT_pearson"],
)
extreme_summary = summarize_extreme_groups(
    unique_factor_df,
    y_col="ridge_RNA_to_ADT_pearson",
)

print("shared latent:", z_source)
print("features:", unique_factor_df.shape[0])
print("valid unique factor ratios:", unique_factor_df["unique_factor_ratio"].notna().sum())
print("correlation summary:", corr_summary)
print("extreme-group summary:", extreme_summary)

unique_factor_df.sort_values("unique_factor_ratio", ascending=False).head()

## 9. Visualize the relationship

In [ ]:
plot_df = unique_factor_df.dropna(subset=["unique_factor_ratio", "ridge_RNA_to_ADT_pearson"]).sort_values("unique_factor_ratio")
rolling_df = ranked_rolling_mean(
    plot_df,
    y_col="ridge_RNA_to_ADT_pearson",
)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].scatter(plot_df["unique_factor_ratio"], plot_df["ridge_RNA_to_ADT_pearson"], s=16, alpha=0.75)
axes[0].set_xlabel("Unique factor ratio")
axes[0].set_ylabel("Ridge RNA-to-ADT Pearson")
axes[0].set_title("Feature-level comparison")

rolling_col = "ridge_RNA_to_ADT_pearson_rolling"
axes[1].plot(np.arange(len(rolling_df)), rolling_df[rolling_col], color="black")
axes[1].set_xlabel("Features ranked by unique factor ratio")
axes[1].set_ylabel("Rolling mean Pearson")
axes[1].set_title("Ranked rolling trend")

plt.tight_layout()
plt.show()

## 10. Result table

The resulting table can be exported or used for downstream interpretation.

In [ ]:
unique_factor_df.head()